# SAMI — Notebook 2 · Behaviour, needs & satisfaction

**Investigative analysis — everything that is not NLP.** Where Notebook 1 asked
*"what data do we have and who is the audience?"*, this notebook investigates
*relationships*: variable cross-cuts, how usage and satisfaction evolve over
time, what users need and where that demand concentrates, and how deeply they
engage before dropping off.

*Clustering, emergent themes, and sentiment/emotion live in Notebook 3.*

---

### Objectives
- Investigate relationships the general profile cannot reveal on its own
  (gender × nationality, engagement by city, age × destination).
- Analyze how chatbot usage and MEAL satisfaction evolve over time.
- Quantify what users need and *where / when* that demand concentrates.
- Measure depth of use and abandonment, and their link to satisfaction.

### Guiding questions
1. What patterns appear only when relationships are investigated
   (gender × nationality, engagement by city, age × destination)?
2. How do usage and satisfaction evolve over time?
3. What do users need most, and where / when does that demand concentrate?
4. Do different profiles ask for different things?
5. How deeply do users engage, and where do they drop off?
6. Which need categories produce the lowest satisfaction?
7. Who responded to the MEAL survey, and what is the general satisfaction pulse?

### Hypotheses
- **H1.** Usage and satisfaction show a trend (rising, or with breaks) aligned to
  project milestones.
- **H2.** Demand concentrates in a few categories (regularization / documentation)
  and a few cities.
- **H3.** Demand peaks align with migration-policy announcements. *(Not tested — we have no dataset of dated policy announcements to align against; see the note in §5.2.)*
- **H4.** Gender or nationality influence which category is asked about most.
- **H5.** Most users drop off after 1–2 messages.
- **H6.** The "regularization / documents" category concentrates the worst MEAL
  ratings.


## Setup

In [1]:
# Imports. Collapsed on purpose -- no analysis here.
# This notebook renders with Plotly (interactive) end-to-end; matplotlib is kept
# only as a colour utility (sampling the brand blue ramp into hex).
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from scipy.stats import gaussian_kde
from matplotlib.colors import LinearSegmentedColormap, to_hex

In [2]:
# ---- Brand palette + Plotly theme --------------------------------------------
# Official SAMI colours (shared with Notebook 1). Blue ("agua") is the workhorse:
# every magnitude chart uses PRIMARY or the blue ramp (light->dark = small->large).
# The accent hues carry identity where slices are genuinely different things;
# "madera" doubles as the neutral for residual "other" categories. A Plotly
# template built from these tokens is registered as the notebook default, so
# re-theming the whole notebook is a matter of editing THIS cell only.
COLORES = {
    "agua":   "#6281DA",   # brand blue (primary)
    "madera": "#8E8E78",   # olive-grey (neutral)
    "hongo":  "#E0DDDC",   # warm light grey (grid / tracks)
    "negro":  "#000000",
    "cielo":  "#EFEFEF",   # light grey (soft surface)
    "ameba":  "#D5A9EA",   # lilac (accent)
    "arbol":  "#6E8149",   # green (accent)
}
AGUA, MADERA, HONGO = COLORES["agua"], COLORES["madera"], COLORES["hongo"]
NEGRO, ARBOL, AMEBA = COLORES["negro"], COLORES["arbol"], COLORES["ameba"]

PRIMARY  = COLORES["agua"]
DARKBLUE = "#26386e"       # dark end of the blue ramp (emphasis)
BLUE_CMAP = LinearSegmentedColormap.from_list(
    "sami_agua", ["#eef1fb", "#c7d1f2", "#9cabe7", "#6281DA", "#41599f", "#26386e"])

def seq_hex(n, lo=0.30, hi=0.95):
    """n blue hex strings, light->dark, for an ordered magnitude (small->large)."""
    if n <= 1:
        return [PRIMARY]
    return [to_hex(BLUE_CMAP(lo + (hi - lo) * i / (n - 1))) for i in range(n)]

def seq_by_value_hex(values, lo=0.22, hi=0.95):
    """One blue hex per value, shade scaled to the value itself (min->max)."""
    v = np.asarray(values, dtype=float)
    if v.size == 0 or v.max() == v.min():
        return [PRIMARY] * len(v)
    t = (v - v.min()) / (v.max() - v.min())
    return [to_hex(BLUE_CMAP(lo + (hi - lo) * ti)) for ti in t]

# Continuous Plotly colorscale sampled from the same ramp (light -> dark blue).
BLUE_SCALE = [[i / 10, to_hex(BLUE_CMAP(i / 10))] for i in range(11)]

# Back-compat aliases used by unchanged cells.
seq_colors = seq_hex
def bar_colors(n):
    return seq_hex(n)
BLUE_SEQ = BLUES = seq_hex(6)

# Categorical identity set, fixed order: agua / arbol / ameba, then madera as the
# neutral residual. Never cycle a 5th hue -- fold extras into madera.
CAT = [COLORES["agua"], COLORES["arbol"], COLORES["ameba"], COLORES["madera"]]
NEUTRAL = COLORES["madera"]
EARTH = CAT

def cat_colors(n):
    """n distinct categorical colours (fixed order, no cycling past the set)."""
    if n > len(CAT):
        return CAT[:-1] + [NEUTRAL] * (n - len(CAT) + 1)
    return CAT[:n]

# Stable gender -> colour map: identity must never depend on a chart's frequency
# ordering, so every gender keeps the same hue across the notebook.
GENDER_COLOR = {"Mujer": CAT[0], "Hombre": CAT[1], "Otro": CAT[2],
                "Prefiero no responder": NEUTRAL}

# Two-hue diverging pair for "above / below the average" deviation charts:
# blue = over-represented, olive = under-represented, neutral grey at zero.
DIV_POS, DIV_NEG = PRIMARY, COLORES["madera"]

# Text / structure tokens.
INK   = COLORES["negro"]
INK2  = "#4d4d4d"
MUTED = COLORES["madera"]
GRID  = COLORES["hongo"]
SURFACE = "white"

# ---- Plotly template (registered as the notebook default) --------------------
pio.templates["sami"] = go.layout.Template(layout=dict(
    font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=13, color=INK),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    colorway=CAT,
    title=dict(font=dict(size=17, color=INK), x=0.02, xanchor="left"),
    xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID,
               ticks="outside", tickcolor=GRID, ticklen=4, automargin=True),
    yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID,
               ticks="outside", tickcolor=GRID, ticklen=4, automargin=True),
    legend=dict(font=dict(size=11), bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=70, r=40, t=64, b=52),
    colorscale=dict(sequential=BLUE_SCALE),
))
pio.templates.default = "sami"

# One date format for every temporal axis ("April 14"); the year lives in the
# chart title so the axis stays uncluttered.
DATE_FMT = "%B %d"

def pct_count_autopct(values, min_pct=3.0):   # kept for API compatibility
    total = float(sum(values))
    def _fmt(pct):
        return "" if pct < min_pct else f"{pct:.1f}%\n({int(round(pct/100*total))})"
    return _fmt

In [3]:
# ---- English display labels --------------------------------------------------
# Every value in the data stays in its original Spanish; ONLY the text drawn on a
# chart is translated, so no plot is ever bilingual. `en(mapping, label)` looks a
# label up and falls back to the label itself when it is already English.
GENDER_EN = {"Mujer": "Woman", "Hombre": "Man",
             "Otro": "Other", "Prefiero no responder": "Prefer not to say"}
MINORS_EN = {"Si": "Yes", "Sí": "Yes", "No": "No"}
NATIONALITY_EN = {"Venezuela": "Venezuela", "Colombia": "Colombia", "Ecuador": "Ecuador",
                  "Perú": "Peru", "Cuba": "Cuba", "Haití": "Haiti", "Chile": "Chile",
                  "Argentina": "Argentina", "España": "Spain", "México": "Mexico",
                  "Panamá": "Panama", "Otros": "Other"}
DURATION_EN = {"Menos de 1 mes": "Less than 1 month", "Entre 1 y 3 meses": "1–3 months",
               "Entre 4 y 6 meses": "4–6 months", "Entre 7 meses y 1 año": "7–12 months",
               "Más de 1 año": "More than 1 year", "No especifica": "Not specified"}
AWAY_EN = {"Menos de 1 mes": "Less than 1 month", "Entre 1 a 3 meses": "1–3 months",
           "Entre 3 meses y 1 año": "3 months – 1 year", "Entre 1 a 5 años": "1–5 years",
           "Hace más de 5 años": "More than 5 years"}
AWAY_ORDER = ["Less than 1 month", "1–3 months", "3 months – 1 year",
              "1–5 years", "More than 5 years"]
RATING_EN = {"Nada útil": "Not useful", "Poco útil": "Slightly useful",
             "Medianamente útil": "Moderately useful", "Útil": "Useful",
             "Muy útil": "Very useful"}
RATING_ORDER_EN = ["Not useful", "Slightly useful", "Moderately useful",
                   "Useful", "Very useful"]
RECOMMEND_EN = {"Sí": "Yes", "No": "No", "Prefiero no responder": "Prefer not to say"}
DISCOVERY_EN = {"Recomendación de otro migrante": "Referral from another migrant",
                "Recomendación de una ONG": "NGO referral", "Redes sociales": "Social media",
                "Cartelera en un punto de atención": "Poster at a service point",
                "Otro": "Other"}
SURVEY_EN = {"Si": "Sent", "Sí": "Sent", "Not sent": "Not sent"}
# Procedures & institutions: acronyms/proper institutions kept, glossed in English;
# descriptive Spanish labels translated.
ENTITY_EN = {"Trámites de documentación": "Documentation procedures",
             "EPS": "EPS (health insurer)", "SISBÉN": "SISBÉN",
             "Migración Colombia": "Migración Colombia (immigration authority)",
             "ACNUR": "UNHCR", "Cancillería": "Foreign Ministry",
             "Registraduría": "Civil Registry", "SENA": "SENA (training agency)",
             "ICBF": "ICBF (family welfare)", "Refugio/Asilo": "Refuge / Asylum",
             "Trabajo/Empleo": "Work / Employment", "Educación": "Education",
             "Vivienda/Arriendo": "Housing / Rent", "Ayuda humanitaria": "Humanitarian aid"}

def en(mapping, label):
    return mapping.get(label, label)


In [4]:
# Data loaders -- inlined so the notebook is fully self-contained (no external module).
import re, unicodedata

DATA_DIR = "../data_&_docs"
RESPONSES_PATH = DATA_DIR + "/MMC_bot_responses_1783087815.xlsx"
MEAL_PATH = DATA_DIR + "/MMC_MEAL_1783087939.xlsx"
DATA_HEADER_ROW = 2  # header is the 3rd row of the export

def _phone(name):
    return re.sub(r"\D", "", str(name))

_CITY_CANON = {
    "medellin": "Medellín", "medellin antioquia": "Medellín", "belen": "Medellín",
    "bogota": "Bogotá", "bogota dc": "Bogotá",
    "cucuta": "Cúcuta",
    "barranquilla": "Barranquilla",
    "santa marta": "Santa Marta",
    "cali": "Cali",
    "cartagena": "Cartagena",
    "bucaramanga": "Bucaramanga",
    "ipiales": "Ipiales",
    "riohacha": "Riohacha", "maicao": "Maicao",
    "soacha": "Soacha", "soacha cundinamarca": "Soacha",
    "necocli": "Necoclí",
}
_NON_CITY = {"colombia", "cundinamarca", "antioquia", "otra", "nan"}

def _fold(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.strip().lower()

def city_canon(name):
    if name is None:
        return "Otra"
    key = _fold(name)
    if key in _NON_CITY or key == "":
        return "Otra"
    if key in _CITY_CANON:
        return _CITY_CANON[key]
    for k, v in _CITY_CANON.items():
        if key.startswith(k):
            return v
    return "Otra"

def clean_city(raw_city, city_other):
    raw = ("" if raw_city is None else str(raw_city)).strip()
    other = "" if city_other is None or pd.isna(city_other) else str(city_other).strip()
    if raw == "Otra" and other:
        return other.title()
    return raw

def _read_whatsapp(path):
    d = pd.read_excel(path, header=DATA_HEADER_ROW)
    d = d[d["Name"].astype(str).str.startswith("whatsapp")].copy()
    d.reset_index(drop=True, inplace=True)
    return d

def load_responses(path=RESPONSES_PATH):
    d = _read_whatsapp(path)
    d["phone"] = d["Name"].map(_phone)
    d["city_clean"] = [clean_city(c, o) for c, o in zip(d["City"], d["City_other"])]
    d["city_canon"] = d["city_clean"].map(city_canon)
    d["age_num"] = pd.to_numeric(d["Age"], errors="coerce")
    d["ts"] = pd.to_datetime(d["Timestamp"], errors="coerce", utc=True).dt.tz_localize(None)
    d["n_questions"] = pd.to_numeric(d["Questions per user"], errors="coerce")
    return d

def load_meal(path=MEAL_PATH):
    d = _read_whatsapp(path)
    d["phone"] = d["Name"].map(_phone)
    cols = list(d.columns)
    d = d.rename(columns={cols[2]: "utility", cols[3]: "would_recommend",
                          cols[4]: "recommendation", cols[5]: "heard_channel",
                          cols[6]: "heard_medium"})
    d["ts"] = pd.to_datetime(d["Timestamp"], errors="coerce", utc=True).dt.tz_localize(None)
    return d

_NOISE = {"undefined", "?", ""}
def _is_noise(t):
    t = t.strip()
    return len(t) < 3 or t.isdigit() or t.lower() in _NOISE

def load_messages(d=None):
    """Explode the per-user `Messages` blob into one row per user turn."""
    if d is None:
        d = load_responses()
    carry = ["phone", "city_clean", "city_canon", "ts", "Gender",
             "Age Ranges", "Nationality", "age_num"]
    carry = [c for c in carry if c in d.columns]
    rows = []
    for _, r in d.iterrows():
        blob = r.get("Messages")
        if not isinstance(blob, str):
            continue
        parts = [p.strip() for p in blob.split("\n")]
        parts = [p for p in parts if not _is_noise(p)]
        for i, p in enumerate(parts):
            row = {c: r[c] for c in carry}
            row["msg_idx"] = i
            row["n_msgs_user"] = len(parts)
            row["message"] = p
            rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)

In [5]:
# Dictionary-based extraction of trámites & institutions -- inlined below (no external module).
from collections import Counter as _Counter

def _ent_norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.lower()

ENTITY_PATTERNS = {
    # Documentation procedures: every paperwork trámite unified into ONE entity --
    # from the user's point of view "sort out my papers" is a single need, and
    # splitting visa/passport/ID/PPT/PEP just fragmented the signal.
    "Trámites de documentación": [
        r"\bppt\b", r"permiso por proteccion temporal",
        r"\bpep\b", r"permiso especial de permanencia",
        r"\bvisa\b", r"\bvisas\b", r"cedula de extranjeria", r"\bpasaporte\b"],
    "EPS": [r"\beps\b", r"afiliacion en salud", r"seguro de salud"],
    "SISBÉN": [r"\bsisben\b"],
    "Migración Colombia": [r"migracion colombia", r"\bmigracion\b"],
    "ACNUR": [r"\bacnur\b", r"\bunhcr\b"],
    "Cancillería": [r"cancilleria"],
    "Registraduría": [r"registraduria"],
    "SENA": [r"\bsena\b"],
    "ICBF": [r"\bicbf\b"],
    "Refugio/Asilo": [r"\brefugio\b", r"\basilo\b", r"solicitante de refugio"],
    "Trabajo/Empleo": [r"\bempleo\b", r"\btrabajo\b", r"permiso de trabajo"],
    "Educación": [r"\beducacion\b", r"\bcolegio\b", r"\bestudios?\b", r"convalidacion"],
    "Vivienda/Arriendo": [r"\barriendo\b", r"\bvivienda\b", r"subsidio de arriendo"],
    "Ayuda humanitaria": [r"ayuda humanitaria", r"asistencia humanitaria"],
}
_COMPILED = {k: [re.compile(p) for p in pats] for k, pats in ENTITY_PATTERNS.items()}

# Split every entity into WHO users must deal with vs WHAT they are trying to do,
# so institutions and procedures are charted separately (never on one axis).
ENTITY_KIND = {
    "Migración Colombia": "institution", "ACNUR": "institution",
    "Cancillería": "institution", "Registraduría": "institution",
    "SENA": "institution", "ICBF": "institution", "EPS": "institution",
    "SISBÉN": "institution",
    "Trámites de documentación": "process", "Refugio/Asilo": "process",
    "Trabajo/Empleo": "process", "Educación": "process",
    "Vivienda/Arriendo": "process", "Ayuda humanitaria": "process",
}

def extract_entities(text):
    t = _ent_norm(text)
    return {name for name, pats in _COMPILED.items() if any(p.search(t) for p in pats)}

def entity_counts(texts):
    c = _Counter()
    for t in texts:
        if t is None or (isinstance(t, float) and pd.isna(t)):
            continue
        for ent in extract_entities(t):
            c[ent] += 1
    return pd.Series(c, dtype="int64").sort_values(ascending=False)

In [6]:
# The 7 official MMC categories, each with a short Spanish hypothesis. The
# category KEYS (English) are what gets displayed on every chart; the VALUES
# stay in Spanish because they are aligned with the bot's original Chat_summary labels. Short labels were chosen by measuring agreement against the bot's
# own Chat_summary labels (see the validation cell in Section 3): they gave the
# best balanced accuracy — long keyword-stuffed descriptions collapsed onto one or two categories.
MMC_LABELS = {
    "legal documentation":     "documentación legal y trámites migratorios",
    "humanitarian assistance": "ayuda humanitaria",
    "employment":              "empleo",
    "services":                "salud y servicios",
    "protection":              "protección y seguridad",
    "journey information":     "información de viaje",
    "organization search":     "búsqueda de una organización",
}
MMC_CATS = list(MMC_LABELS)
COURTESY = "courtesy / non-substantive"
NOISE    = "unclustered"
FRAG     = "short fragment"                 # location/greeting fragments, not a theme
NON_TOPIC_CATS = [COURTESY, NOISE, FRAG]    # excluded from topic cross-cuts

# one distinct colour per category (brand colours first, then muted extras)
CAT_COLORS = {
    "legal documentation":     AGUA,
    "humanitarian assistance": ARBOL,
    "employment":              AMEBA,
    "services":                MADERA,
    "protection":              DARKBLUE,
    "journey information":     "#9cabe7",
    "organization search":     "#47552f",
    "emergent":                "#b98fd0",
    COURTESY:                  HONGO,
    NOISE:                     "#cfcfcf",
    FRAG:                      "#b0b0b0",
}
def cat_palette(cats):
    return [CAT_COLORS.get(c, NEGRO) for c in cats]

### Data

`df` / `meal` come from the same cleaning as Notebook 1 (rich display columns).
`msgs` is the message-level spine; `_meal_m` is the MEAL survey keyed by phone
(for the satisfaction × category cut). `mmc_category` is the DB's own
`Chat_summary` classification — the non-NLP original taxonomy.

In [7]:
DATA_PATH = '../data_&_docs/MMC_bot_responses_1783087815.xlsx'

df = pd.read_excel(DATA_PATH, sheet_name='mmc bot - responses', header=2)
df = df.dropna(how='all').reset_index(drop=True)

# One row has no Name/Timestamp/other field except a stray "Questions per
# user" = 388 -- a spreadsheet artifact, not a real interaction. Drop it.
df = df[df['Name'].notna()].reset_index(drop=True)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Questions per user'] = pd.to_numeric(df['Questions per user'], errors='coerce')

# Consolidate _other columns: fall back to free-text when main option is NaN/"Otra"
df['city_display'] = df.apply(
    lambda r: r['City_other'] if r['City'] == 'Otra' else r['City'], axis=1
)
df['nationality_display'] = df.apply(
    lambda r: r['Nationality_other'] if pd.isna(r['Nationality']) else r['Nationality'], axis=1
)

# Canonicalize city: city_display is very messy -- the same city appears under
# many spellings ("Bogotá"/"Bogota"/"Bogotá D.C"/"Colombia Bogotá"), with mixed
# case ("Santa marta"), and with department suffixes ("Soacha Cundinamarca",
# "Barranquilla Atlántico"). We match on an accent/case-insensitive key and map
# every variant to one canonical name, and drop entries that are departments or
# a country rather than a city ("Colombia", "Cundinamarca", "Antioquia", "9").
import unicodedata

def _city_key(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s.lower().strip().rstrip('.').strip()

CITY_CANON_KEYS = {
    'medellin': 'Medellín',
    'bogota': 'Bogotá', 'bogota dc': 'Bogotá', 'bogota d.c': 'Bogotá',
    'bogota d c': 'Bogotá', 'colombia bogota': 'Bogotá', 'bogota zipaquira': 'Bogotá',
    'cucuta': 'Cúcuta',
    'santa marta': 'Santa Marta',
    'soacha': 'Soacha', 'soacha cundinamarca': 'Soacha', 'soacha condinamarca': 'Soacha',
    'soacha, cundinamarca': 'Soacha',
    'cali': 'Cali',
    'necocli': 'Necoclí',
    'barranquilla': 'Barranquilla', 'barranquilla atlantico': 'Barranquilla',
    'riohacha': 'Riohacha', 'riohacha la guajira': 'Riohacha',
    'turbo': 'Turbo', 'turbo antioquia': 'Turbo',
    'ipiales': 'Ipiales', 'bucaramanga': 'Bucaramanga', 'maicao': 'Maicao',
    'cartagena': 'Cartagena', 'tumaco': 'Tumaco', 'pasto': 'Pasto',
    'pereira': 'Pereira', 'galapa': 'Galapa',
}
CITY_DROP = {'9', 'colombia', 'cundinamarca', 'antioquia'}

def _canon_city(c):
    if not isinstance(c, str):
        return c
    k = _city_key(c)
    if k in CITY_DROP:
        return np.nan
    return CITY_CANON_KEYS.get(k, c.strip())

df['city_display'] = df['city_display'].map(_canon_city)

# Canonicalize nationality: merge duplicate spellings of the same country so
# e.g. "Colombia" and "Colombiana" are ONE category, and drop non-country junk.
NATIONALITY_CANON = {
    'Colombiana': 'Colombia', 'Soy colombovenezolana': 'Venezuela',
    'Haiti': 'Haití', 'Panama': 'Panamá', 'Peru': 'Perú', 'Mexico': 'México',
}
INVALID_NATIONALITIES = {'1', '3', '4', 'Valyria'}
_nat = df['nationality_display'].astype('string').str.strip().replace(NATIONALITY_CANON)
df['nationality_clean'] = _nat.where(~_nat.isin(INVALID_NATIONALITIES))

# City_duration is free-text with ~20 spelling/format variants of the same 5
# duration ranges. DURATION_MAP normalizes to 5 ranges + "No especifica".
DURATION_ORDER = [
    'Menos de 1 mes', 'Entre 1 y 3 meses', 'Entre 4 y 6 meses',
    'Entre 7 meses y 1 año', 'Más de 1 año', 'No especifica',
]
DURATION_MAP = {
    'Más de 1 año': 'Más de 1 año', 'Menos de 1 mes': 'Menos de 1 mes',
    'Entre 1 y 3 meses': 'Entre 1 y 3 meses',
    'Entre 7 meses y 1 año': 'Entre 7 meses y 1 año',
    'Entre 4 y 6 meses': 'Entre 4 y 6 meses',
    '2 años': 'Más de 1 año', '15 días': 'Menos de 1 mes', '8 años': 'Más de 1 año',
    '5 años': 'Más de 1 año', '3 años': 'Más de 1 año',
    'Ya tengo una semana': 'Menos de 1 mes', '7 años': 'Más de 1 año',
    'Vendo': 'No especifica', '8ños': 'Más de 1 año', '7 años 9 meses': 'Más de 1 año',
    '5 anos': 'Más de 1 año', '2 año': 'Más de 1 año', 'Santuario': 'No especifica',
    'Acabo de llegar': 'Menos de 1 mes', '3 meses': 'Entre 1 y 3 meses',
    'Norte de Santander': 'No especifica',
}
df['city_duration_clean'] = df['City_duration'].map(DURATION_MAP)

print(f"Shape: {df.shape}")
df.dtypes

Shape: (947, 33)


Name                                   str
Subitems                           float64
Timestamp              datetime64[us, UTC]
Consent                                str
Nationality                            str
Nationality_other                      str
City                                   str
City Location                      float64
City_other                             str
City_duration                          str
Gender                                 str
Gender_other                           str
Age                                float64
Minors                                 str
Away_duration                          str
Prev_country                       float64
Prev_country_other                     str
Destination                            str
Destination_other                      str
Destination_Country                    str
Messages                               str
Survey sent                            str
Chat_summary                           str
Summarize  

In [8]:
# Source spreadsheets: MEAL feedback form and chatbot interaction log
MEAL_PATH = '../data_&_docs/MMC_MEAL_1783087939.xlsx'
RESP_PATH = '../data_&_docs/MMC_bot_responses_1783087815.xlsx'

# header=2: the sheet has two banner rows above the real column headers
meal = pd.read_excel(MEAL_PATH, sheet_name='mmc-meal', header=2)
meal = meal.dropna(how='all').reset_index(drop=True)  # drop blank rows left by the banner

# Replace the long Spanish question text with short, code-friendly column names
meal.columns = [
    'Name', 'Timestamp',
    'usefulness_rating', 'would_recommend',
    'recommendation_text', 'discovery_channel', 'discovery_other',
]
meal['Timestamp'] = pd.to_datetime(meal['Timestamp'], errors='coerce')

# Map the Spanish Likert labels to a 1-5 numeric scale so we can average / trend
RATING_MAP = {
    'Muy útil': 5, 'Útil': 4,
    'Medianamente útil': 3, 'Poco útil': 2, 'Nada útil': 1,
}
meal['rating_num'] = meal['usefulness_rating'].map(RATING_MAP)

print(f"MEAL records: {len(meal)}")
meal.head()

MEAL records: 78


,Name,Timestamp,usefulness_rating,would_recommend,recommendation_text,discovery_channel,discovery_other,rating_num
0,whatsapp:+56929572884,2026-03-26 00:47:35.583000+00:00,Muy útil,Sí,Excelente,Recomendación de una ONG,NaN,5
1,whatsapp:+573113165608,2026-03-26 10:12:14.030000+00:00,Nada útil,Prefiero no responder,"Ayuda, no se cómo activar mi línea nueva Tigo ...",Otro,Redes sociales,1
2,whatsapp:+16462943119,2026-03-26 16:06:16.380000+00:00,Nada útil,Prefiero no responder,No recibí ninguna información.,Redes sociales,NaN,1
3,whatsapp:+573192311911,2026-03-27 03:36:50.975000+00:00,Medianamente útil,Prefiero no responder,No,Recomendación de una ONG,NaN,3
4,whatsapp:+16462943119,2026-03-27 11:47:53.350000+00:00,Nada útil,Prefiero no responder,No recibí información,Otro,Otro,1


In [9]:
# Message spine + phone-keyed MEAL, from the inlined loaders above.
_resp = load_responses()
msgs = load_messages(_resp)
_meal_m = load_meal()
print(f"df users: {len(df)}  |  meal: {len(meal)}  |  messages: {len(msgs)}")

df users: 947  |  meal: 78  |  messages: 2993


In [10]:
# Original classification from the bot's Chat_summary (non-NLP): one dominant
# category per user, broadcast to their messages. Ground truth for the cross-cuts
# (NB2) and the cluster-vs-original check + tinting (NB3).
TOPIC_MAP_LC = {
    "legal documentation":     "legal documentation",
    "humanitarian assistance": "humanitarian assistance",
    "employment":              "employment",
    "services":                "services",
    "protection":              "protection",
    "organization search":     "organization search",
    "journey information":     "journey information",
}
UNCLASSIFIED = "unclassified"

def _user_category(summary):
    if not isinstance(summary, str) or "Use exactly one of these hashtags" in summary:
        return UNCLASSIFIED
    cats = []
    for t in summary.replace("''", ",").split(","):
        t = t.strip().lstrip("#").replace("_", " ").lower()
        if t in TOPIC_MAP_LC:
            cats.append(TOPIC_MAP_LC[t])
    if not cats:
        return UNCLASSIFIED
    return pd.Series(cats).value_counts().index[0]

_resp["mmc_category"] = _resp["Chat_summary"].map(_user_category)

def _dominant_cat(s):
    known = [c for c in s if c != UNCLASSIFIED]
    return pd.Series(known).value_counts().index[0] if known else UNCLASSIFIED

# phone can repeat -> aggregate to one dominant category per user
_cat_by_phone = _resp.groupby("phone")["mmc_category"].agg(_dominant_cat)
msgs["mmc_category"] = msgs["phone"].map(_cat_by_phone).fillna(UNCLASSIFIED)

CAT_COLORS[UNCLASSIFIED] = "#cfcfcf"
NON_TOPIC_CATS = NON_TOPIC_CATS + [UNCLASSIFIED]
print(msgs["mmc_category"].value_counts())

mmc_category
legal documentation        1552
humanitarian assistance     739
employment                  322
services                    254
organization search          60
journey information          29
protection                   25
unclassified                 12
Name: count, dtype: int64


## 1. Strategic cross-cuts

**What this shows.** Patterns that only appear when variables are crossed —
invisible in the one-variable-at-a-time profile of Notebook 1. Each cut is
annotated with sample size so thin slices are not over-read.

### 1.1 How settled are users, city by city?

**What this shows.** For each major city, the share of recent arrivals vs
long-settled residents. A 100% stacked bar makes the *mix* comparable across
cities regardless of sample size.

In [11]:
# For each major city, what share are recent arrivals vs long-settled? Only the
# Top-5 cities by user count are shown individually; every smaller city is folded
# into a single "Other" column so the comparison stays legible. A 100% stacked bar
# makes the *mix* comparable across cities regardless of sample size.
TOP_N = 5
counts_city = df['city_display'].value_counts()
top_cities = counts_city.head(TOP_N).index.tolist()
grp = df.dropna(subset=['city_display', 'city_duration_clean']).copy()
grp['city_grp'] = grp['city_display'].where(grp['city_display'].isin(top_cities), 'Other')

order_rows = top_cities + ['Other']
cross = (grp.groupby(['city_grp', 'city_duration_clean']).size()
         .unstack(fill_value=0).reindex(order_rows).fillna(0))
cols = [c for c in DURATION_ORDER if c in cross.columns]
cross = cross[cols]
totals = cross.sum(axis=1)
cross_pct = cross.div(totals, axis=0) * 100

# Ordered light->dark blue ramp for the real duration buckets; neutral for the
# residual "No especifica".
real = [c for c in cols if c != 'No especifica']
ramp = seq_hex(len(real))
dur_color = {c: ramp[i] for i, c in enumerate(real)}
dur_color['No especifica'] = MUTED

fig = go.Figure()
for c in cols:
    fig.add_bar(
        y=order_rows, x=cross_pct[c].reindex(order_rows).values, orientation='h',
        name=en(DURATION_EN, c), marker_color=dur_color[c],
        marker_line_color=SURFACE, marker_line_width=1.2,
        hovertemplate="%{y}<br>" + en(DURATION_EN, c) + ": %{x:.0f}%<extra></extra>")
# n= behind each city, just past the 100% line
for city in order_rows:
    fig.add_annotation(x=101, y=city, text=f"n={int(totals[city])}", showarrow=False,
                       xanchor='left', font=dict(size=11, color=MUTED))

fig.update_layout(
    barmode='stack',
    title="How settled are users, city by city?<br>"
          "<span style='font-size:13px;color:#8E8E78'>Share by length of residence · Top 5 cities + Other</span>",
    xaxis=dict(title="Share of users (%)", range=[0, 108], ticksuffix="%"),
    yaxis=dict(title="", autorange='reversed'),
    legend=dict(title="Time in city", orientation='h', y=-0.32, x=0, traceorder='normal'),
    height=470, margin=dict(l=110, r=60, t=90, b=120))
fig.show()

### 1.2 Gender composition by nationality

**What this shows.** Whether gender balance differs by nationality (Top 5 +
Other). A 100% stacked bar shows the *composition* within each group, readable
even where raw counts are tiny.

In [12]:
# Does gender balance differ by nationality? A 100% stacked bar shows the gender
# *composition* within each nationality (Top 5 + Other) -- readable even where the
# raw counts are tiny.
top_nat = df['nationality_clean'].value_counts().head(5).index.tolist()
tmp = df.dropna(subset=['Gender', 'nationality_clean']).copy()
tmp['nat_grp'] = tmp['nationality_clean'].where(tmp['nationality_clean'].isin(top_nat), 'Otros')
order = top_nat + ['Otros']
comp = tmp.groupby(['nat_grp', 'Gender']).size().unstack(fill_value=0).reindex(order).fillna(0)
totals = comp.sum(axis=1)
comp_pct = comp.div(totals, axis=0) * 100
genders = list(comp.columns)
gcolor = {g: GENDER_COLOR.get(g, NEUTRAL) for g in genders}

fig = go.Figure()
for g in genders:
    fig.add_bar(
        y=[en(NATIONALITY_EN, n) for n in order], x=comp_pct[g].values, orientation='h',
        name=en(GENDER_EN, g), marker_color=gcolor[g],
        marker_line_color=SURFACE, marker_line_width=1.2,
        hovertemplate="%{y}<br>" + en(GENDER_EN, g) + ": %{x:.0f}%<extra></extra>")
for i, n in enumerate(order):
    fig.add_annotation(x=101, y=en(NATIONALITY_EN, n), text=f"n={int(totals[n])}",
                       showarrow=False, xanchor='left', font=dict(size=11, color=MUTED))
fig.update_layout(
    barmode='stack',
    title="Gender composition within each nationality<br>"
          "<span style='font-size:13px;color:#8E8E78'>Top 5 nationalities + Other</span>",
    xaxis=dict(title="Share of users (%)", range=[0, 108], ticksuffix="%"),
    yaxis=dict(title="", autorange='reversed'),
    legend=dict(title="Gender", orientation='h', y=-0.30, x=0, traceorder='normal'),
    height=460, margin=dict(l=110, r=60, t=90, b=110))
fig.show()

### 1.3 Engagement by city

**What this shows.** Which cities have the most engaged users (average questions
per user). Each bar carries `n=` so thin samples are not over-read.

In [13]:
# Which cities have the most engaged users (avg questions per user)? Top 5 cities;
# each bar shaded by its own value (light->dark blue) and annotated with n= (users
# behind the average). A dashed line marks the overall average across all users so
# each city reads as above / below the norm.
top_cities = df['city_display'].value_counts().head(5).index
city_eng = (df[df['city_display'].isin(top_cities)]
            .groupby('city_display')['Questions per user']
            .agg(mean='mean', n='count').sort_values('mean', ascending=False))
overall_mean = df['Questions per user'].mean()

fig = go.Figure()
fig.add_bar(
    x=city_eng.index, y=city_eng['mean'],
    marker_color=seq_by_value_hex(city_eng['mean'].values),
    text=[f"{m:.1f}<br><span style='color:#8E8E78'>n={n}</span>"
          for m, n in zip(city_eng['mean'], city_eng['n'])],
    textposition='outside', cliponaxis=False,
    hovertemplate="%{x}<br>avg %{y:.2f} questions/user<extra></extra>")
fig.add_hline(y=overall_mean, line_dash='dash', line_color=MUTED, line_width=1.5,
              annotation_text=f"overall avg {overall_mean:.1f}",
              annotation_position="top right",
              annotation_font=dict(size=11, color=MUTED))
fig.update_layout(
    title="Which cities have the most engaged users?<br>"
          "<span style='font-size:13px;color:#8E8E78'>Average questions per user · Top 5 cities</span>",
    yaxis=dict(title="Avg questions per user", range=[0, city_eng['mean'].max() * 1.28]),
    xaxis=dict(title=""),
    showlegend=False, height=440, margin=dict(t=90))
fig.show()

### 1.4 Age by intended destination

**What this shows.** Whether the age profile differs by intended destination
country (top 5 destinations).

In [14]:
# Does the age profile differ by intended destination country? A box per
# destination (Top 5) shows median, spread and outliers side by side.
top_dests = df['Destination_Country'].value_counts().head(5).index.tolist()
colors = cat_colors(len(top_dests))
fig = go.Figure()
for d, c in zip(top_dests, colors):
    ages = df.loc[df['Destination_Country'] == d, 'Age'].dropna()
    fig.add_box(y=ages, name=str(d), marker_color=c, line_color=c,
                boxmean=True, fillcolor=c, opacity=0.75,
                marker=dict(size=4, opacity=0.5),
                hovertemplate="%{y} yrs<extra>" + str(d) + "</extra>")
fig.update_layout(
    title="Does age differ by intended destination?<br>"
          "<span style='font-size:13px;color:#8E8E78'>Age distribution · Top 5 destinations</span>",
    yaxis=dict(title="Age (years)"), xaxis=dict(title=""),
    showlegend=False, height=460, margin=dict(t=90))
fig.show()

## 2. Trends over time

**What this shows — and a caution.** Two *different* things evolve here, and they
must never be read as one:

- **Reach** — how many users (and MEAL responses) accumulate over time. A rising
  cumulative curve means the bot is *reaching more people*.
- **Satisfaction** — how a given user rates Sami. This is an *individual* property:
  whether one person finds the bot useful, completes the conversation and fills in
  the survey without dropping off.

> **More users does not mean more satisfied users.** Growth in reach (§2.1, and the
> left panel of §2.2) is tracked *separately* from the satisfaction trend (right
> panel of §2.2). The two are never merged into a single "success" line — a curve
> that climbs because more people arrive says nothing about whether they were
> satisfied.

### 2.1 Daily chatbot usage

*Adoption over time — a **behavioural** signal (when people actually reach for
Sami), not a static head-count.* Daily interaction volume: growth, drop-offs and
spikes. This is reach, **not** satisfaction (see the caution above).

In [15]:
# How has daily chatbot usage evolved (growth, drop-offs, spikes)?
timeline = df.dropna(subset=['Timestamp']).set_index('Timestamp').resample('D').size()
yr = timeline.index.year.min()

fig = go.Figure()
fig.add_scatter(x=timeline.index, y=timeline.values, mode='lines',
                line=dict(color=PRIMARY, width=2), fill='tozeroy',
                fillcolor='rgba(98,129,218,0.28)',
                hovertemplate="%{x|%B %d}<br>%{y} interactions<extra></extra>")
fig.update_layout(
    title=f"Daily user interactions over time <span style='color:#8E8E78'>({yr})</span>",
    xaxis=dict(title="", tickformat=DATE_FMT),
    yaxis=dict(title="Interactions"),
    height=380, margin=dict(t=64))
fig.show()

### 2.2 MEAL responses over time

Two panels, deliberately kept apart:

- **Left — cumulative reach.** How many MEAL responses have been collected to date.
  This measures *participation growing*, nothing about how good the experience was.
- **Right — satisfaction trend.** Individual usefulness ratings over time, with a
  smoothed mean — the actual *satisfaction* signal.

> **Do not conflate them.** A climbing reach curve (left) and a flat or falling
> satisfaction trend (right) can happen at the same time. Reaching more users tells
> you nothing about whether those users were satisfied; read each panel on its own.

In [16]:
# Left: cumulative reach of the survey. Right: individual ratings over time with a
# smoothed satisfaction curve (14-day time-weighted mean, drawn as a spline) so the
# trend reads cleanly through the scatter.
timeline = meal.dropna(subset=['Timestamp']).set_index('Timestamp').resample('D').size()
cumulative = timeline.cumsum()
yr = cumulative.index.year.min()

meal_sorted = meal.dropna(subset=['Timestamp', 'rating_num']).sort_values('Timestamp')
# Smooth on a regular daily grid so the trend curve is clean, not spiky: daily mean
# rating -> interpolate gaps -> centred 21-day rolling mean, drawn as a spline.
daily_rating = meal_sorted.set_index('Timestamp')['rating_num'].resample('D').mean()
smooth = daily_rating.interpolate('time').rolling(21, center=True, min_periods=4).mean().dropna()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.09,
                    subplot_titles=("Feedback collected over time (cumulative)",
                                    "Is satisfaction trending up or down?"))

fig.add_scatter(x=cumulative.index, y=cumulative.values, mode='lines', row=1, col=1,
                line=dict(color=PRIMARY, width=2), fill='tozeroy',
                fillcolor='rgba(98,129,218,0.28)', showlegend=False,
                hovertemplate="%{x|%B %d}<br>%{y} responses<extra></extra>")

fig.add_scatter(x=meal_sorted['Timestamp'], y=meal_sorted['rating_num'], mode='markers',
                row=1, col=2, name='Individual rating',
                marker=dict(color=PRIMARY, size=8, opacity=0.55,
                            line=dict(color=SURFACE, width=0.8)),
                hovertemplate="%{x|%B %d}<br>rating %{y}<extra></extra>")
fig.add_scatter(x=smooth.index, y=smooth.values, mode='lines', row=1, col=2,
                name='Smoothed trend', line=dict(color=ARBOL, width=3, shape='spline'),
                hovertemplate="%{x|%B %d}<br>mean %{y:.2f}<extra></extra>")

fig.update_xaxes(tickformat=DATE_FMT, title="")
fig.update_yaxes(title="Cumulative responses", row=1, col=1)
fig.update_yaxes(title="Usefulness rating (1–5)", range=[0.5, 5.5],
                 tickvals=[1, 2, 3, 4, 5], row=1, col=2)
fig.update_layout(
    title=f"MEAL over time — reach (left) is NOT satisfaction (right) <span style='color:#8E8E78'>({yr})</span>",
    legend=dict(orientation='h', y=-0.16, x=0.55, xanchor='center'),
    width=1300, height=470, margin=dict(t=90))
fig.show()

## 3. Who answered MEAL, and the satisfaction pulse

**What this shows.** The satisfaction baseline: who left feedback and how they
rate Sami. Response rate is low (see Notebook 1), so read these as indicative,
not representative — the general *pulse*, not a precise measurement.

### 3.1 Usefulness rating

In [17]:
# How useful did users find Sami? A single 100%-stacked bar over the ordered rating
# scale (least -> most useful, light -> dark blue). Far more legible than the old
# 10x10 waffle -- one row, read left to right, with the headline stats in the title.
rating_order = ['Nada útil', 'Poco útil', 'Medianamente útil', 'Útil', 'Muy útil']
rc = meal['usefulness_rating'].value_counts().reindex(rating_order, fill_value=0)
total = int(rc.sum())
ramp = seq_hex(len(rating_order))                      # light (least) -> dark (most)
mean_r = meal['rating_num'].mean()
useful_plus = int(rc[['Útil', 'Muy útil']].sum())

fig = go.Figure()
for r, col in zip(rating_order, ramp):
    v = int(rc[r]); pct = v / total * 100
    fig.add_bar(y=['Usefulness'], x=[pct], orientation='h', name=en(RATING_EN, r),
                marker=dict(color=col, line=dict(color=SURFACE, width=1.2)),
                text=[f"{pct:.0f}%" if pct >= 4 else ""], textposition='inside',
                insidetextanchor='middle', textfont=dict(color=SURFACE, size=13),
                hovertemplate=f"{en(RATING_EN, r)}: {v} ({pct:.0f}%)<extra></extra>")
fig.update_layout(
    barmode='stack',
    title="How useful did users find Sami?<br>"
          f"<span style='font-size:13px;color:#8E8E78'>{total} rated responses · mean {mean_r:.2f}/5 · "
          f"{useful_plus/total*100:.0f}% rated it Useful or Very useful</span>",
    xaxis=dict(title="Share of responses (%)", range=[0, 100]),
    yaxis=dict(title="", showticklabels=False),
    legend=dict(orientation='h', y=-0.4, x=0.5, xanchor='center', traceorder='normal'),
    height=270, margin=dict(t=95, b=80))
fig.show()

### 3.2 Would recommend

In [18]:
# Would users recommend Sami? A Sankey links each recommendation (left) to how those
# same respondents rated the bot's usefulness (right), so the flow shows e.g. that
# most "Yes" answers came from people who found it very useful.
rec_df = meal.dropna(subset=['would_recommend']).copy()
rec_df['rec_en'] = rec_df['would_recommend'].map(lambda x: en(RECOMMEND_EN, x))
rec_df['rate_en'] = rec_df['usefulness_rating'].map(lambda x: en(RATING_EN, x)).fillna('No rating')

rec_nodes = [en(RECOMMEND_EN, k) for k in meal['would_recommend'].value_counts().index]
rate_nodes = [r for r in RATING_ORDER_EN if r in set(rec_df['rate_en'])]
if 'No rating' in set(rec_df['rate_en']):
    rate_nodes = rate_nodes + ['No rating']
nodes = rec_nodes + rate_nodes
idx = {n: i for i, n in enumerate(nodes)}

rec_color = {'Yes': PRIMARY, 'No': COLORES['ameba'], 'Prefer not to say': MUTED}
rate_ramp = dict(zip(RATING_ORDER_EN, seq_hex(len(RATING_ORDER_EN))))
node_colors = ([rec_color.get(n, NEUTRAL) for n in rec_nodes]
               + [rate_ramp.get(n, HONGO) for n in rate_nodes])

flow = rec_df.groupby(['rec_en', 'rate_en']).size().reset_index(name='n')
def _rgba(hex_c, a=0.45):
    h = hex_c.lstrip('#'); r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{a})"

n_yes = (rec_df['rec_en'] == 'Yes').mean() * 100
fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(label=nodes, color=node_colors, pad=18, thickness=20,
              line=dict(color=SURFACE, width=1.5)),
    link=dict(
        source=[idx[s] for s in flow['rec_en']],
        target=[idx[t] for t in flow['rate_en']],
        value=flow['n'].tolist(),
        color=[_rgba(rec_color.get(s, NEUTRAL)) for s in flow['rec_en']],
        hovertemplate="%{source.label} → %{target.label}<br>%{value} respondents<extra></extra>")))
fig.update_layout(
    title="Would users recommend Sami to someone else?<br>"
          f"<span style='font-size:13px;color:#8E8E78'>Recommendation (left) → usefulness rating (right) · "
          f"{n_yes:.0f}% would recommend</span>",
    font=dict(size=13), height=500, margin=dict(t=100, b=20, l=10, r=10))
fig.show()

### 3.3 Discovery channel

In [19]:
# How did users discover Sami? A plain, ranked horizontal bar. A funnel implies a
# drop-off sequence between the channels that does not exist -- these are just
# independent counts of how each user first heard about the bot.
disc = meal['discovery_channel'].value_counts().sort_values()
labels = [en(DISCOVERY_EN, k) for k in disc.index]
total = int(disc.sum())
fig = go.Figure(go.Bar(
    x=disc.values, y=labels, orientation='h',
    marker=dict(color=seq_by_value_hex(disc.values), line=dict(color=SURFACE, width=0.5)),
    text=[f"{v} ({v/total*100:.0f}%)" for v in disc.values],
    textposition='outside', cliponaxis=False,
    hovertemplate="%{y}<br>%{x} responses<extra></extra>"))
fig.update_layout(
    title="How did users discover Sami?<br>"
          f"<span style='font-size:13px;color:#8E8E78'>Discovery channel · {total} responses</span>",
    xaxis=dict(title="Responses", range=[0, disc.max() * 1.16]),
    yaxis=dict(title=""), height=420, margin=dict(l=220, t=90))
fig.show()

### 3.4 Did users leave written feedback?

*A light engagement check — **what** users write is analysed in Notebook 3 (§6),
not here.* Recommendation **length** on its own says little, so this section is
deliberately minimal: it only confirms how many users bothered to write, then
hands the actual content off to the thematic read in NB3.

In [20]:
# De-emphasised on purpose (see the heading). How MUCH users write says little;
# WHAT they write is analysed thematically in Notebook 3 (§6.1 word cloud, §6.2
# thematic read). Here we keep only a one-line engagement summary -- no chart.
rec_text = meal['recommendation_text'].dropna()
lengths = rec_text.str.len()
left_pct = meal['recommendation_text'].notna().mean() * 100
print(f"{len(rec_text)} of {len(meal)} respondents ({left_pct:.0f}%) left a written recommendation.")
print(f"Median length {lengths.median():.0f} characters (mean {lengths.mean():.0f}).")
print("What they actually wrote -> analysed in Notebook 3, §6.")

77 of 78 respondents (99%) left a written recommendation.
Median length 3 characters (mean 16).
What they actually wrote -> analysed in Notebook 3, §6.


## 4. Most-requested needs (message level)

**What this shows.** What users actually ask for, by dictionary matching of
procedures & institutions across every message (not NLP clustering), and where
that demand concentrates geographically (**H2**).

### 4.1 What users need: institutions vs. procedures

Two dictionary-based reads over the **free-text messages** (not the bot's own
labels), deliberately split into two charts:

- **Institutions** — the named organisations users must actually reach (Migración
  Colombia, UNHCR, SENA, ICBF, the Foreign Ministry, the Civil Registry…). This
  answers *who* the migration journey forces people to deal with.
- **Procedures & needs** — what users are trying to *do* (regularise their
  documents, seek asylum, find work, schooling, housing, humanitarian aid).

Keeping them apart stops an *organisation* being ranked on the same axis as a
*trámite*, which is what made the old single chart hard to read.

**Grouping.** The paperwork items the bot listed separately — *visa, passport,
foreigner ID, PPT, PEP* — are unified into a single **"Documentation procedures"**
entity: from the user's point of view these are one need ("sort out my papers"),
and splitting them only fragmented the signal.

> **Why this is behaviour, not description.** We are not re-tabulating the bot's
> pre-assigned categories here — we mine what users *spontaneously wrote* to infer
> the institutions and trámites their situation demands. That is a step toward the
> service map MMC would need, not a restatement of the input data.

In [21]:
# Two separate reads of the message text: WHICH INSTITUTIONS users name (who they
# must deal with) and WHICH PROCEDURES they raise (what they are trying to do).
# Splitting them keeps an organisation from being compared against a trámite.
counts = entity_counts(msgs["message"])

def _entity_bar(kind, title, subtitle):
    ents = [e for e in counts.index if ENTITY_KIND.get(e) == kind]
    s = counts.reindex(ents).dropna().astype(int).sort_values()
    labels = [en(ENTITY_EN, e) for e in s.index]
    fig = go.Figure(go.Bar(
        x=s.values, y=labels, orientation="h",
        marker=dict(color=seq_by_value_hex(s.values), line=dict(color=SURFACE, width=0.5)),
        text=s.values, textposition="outside", cliponaxis=False,
        hovertemplate="%{y}<br>%{x} messages<extra></extra>"))
    fig.update_layout(
        title=f"{title}<br><span style='font-size:13px;color:#8E8E78'>{subtitle}</span>",
        xaxis=dict(title="Messages mentioning", range=[0, s.max() * 1.14]),
        yaxis=dict(title=""), height=110 + 34 * len(s), margin=dict(l=250, t=90))
    fig.show()

_entity_bar("institution", "Institutions users need to reach",
            "Named organisations mentioned in messages · shaded by volume")
_entity_bar("process", "Procedures &amp; needs users raise",
            "Trámites and needs mentioned in messages · shaded by volume")

# per-entity boolean columns for cross-tabs
for ent in counts.index:
    msgs["ent_" + ent] = msgs["message"].fillna("").map(
        lambda t, e=ent: e in extract_entities(t))

### 4.2 Messages by city

In [22]:
# Messages by city as a treemap: tile area is message volume, shaded in the NEUTRAL
# olive-grey ramp (light->dark = small->large). City is a *magnitude*, so it is kept
# out of the brand identity hues -- that way city tiles never collide with the
# coloured CATEGORY tiles in §5.1 (categories own the colours; cities own the greys).
top_cities = msgs.loc[msgs["city_canon"] != "Otra", "city_canon"].value_counts().head(10)
tot = int(top_cities.sum())
tc = top_cities.reset_index()
tc.columns = ["city", "messages"]

# Neutral olive-grey ramp, medium->dark so white labels stay legible on every tile.
GREY_SCALE = [[0.0, "#a8a89a"], [0.5, "#78786a"], [1.0, "#43433a"]]

fig = px.treemap(tc, path=["city"], values="messages",
                 color="messages", color_continuous_scale=GREY_SCALE)
fig.update_traces(
    marker=dict(cornerradius=6, line=dict(color=SURFACE, width=3)),
    texttemplate="<b>%{label}</b><br>%{value} msgs · %{percentRoot:.0%}",
    textfont=dict(size=15, color=SURFACE), textposition="middle center",
    hovertemplate="%{label}<br>%{value} messages (%{percentRoot:.1%})<extra></extra>")
fig.update_layout(
    title="Messages by city<br>"
          f"<span style='font-size:13px;color:#8E8E78'>Top 10 · {tot:,} messages · tile size &amp; grey shade = volume</span>",
    coloraxis_showscale=False, height=540, margin=dict(t=90, l=10, r=10, b=10))
fig.show()

## 5. Needs by category (original classification)

**What this shows.** The named view of *demand*, built on the DB's own
`Chat_summary` categories: how the need mix shifts **by city, over time, and by
profile**, where users drop off, and which needs rate worst. The aim is not to
re-describe the categories (that is Notebook 1's job) but to read them as
**behavioural demand signals** — which needs concentrate where, for whom, and with
what satisfaction — that feed the service and architecture recommendations
(**H2**, **H4**, **H5**, **H6**).

### 5.1 Category mix by city

In [23]:
# Needs by category, city by city, as a nested treemap: each city block (Top 5 by
# message volume + Other) is sized by its share of messages and subdivided into its
# category mix. CATEGORY tiles use the brand CAT_COLORS; the CITY parent blocks are
# forced to a neutral grey so the two levels never share a hue (the collision Diana
# flagged) -- colour = category, grey = city, consistently with the grey city treemap in 4.2.
topic = msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)].copy()
top5 = topic["city_canon"].value_counts()
top5 = [c for c in top5.index if c != "Otra"][:5]
topic = topic[topic["city_canon"] != "Otra"].copy()
topic["city_grp"] = topic["city_canon"].where(topic["city_canon"].isin(top5), "Other")

agg = topic.groupby(["city_grp", "mmc_category"]).size().reset_index(name="n")

fig = px.treemap(agg, path=["city_grp", "mmc_category"], values="n",
                 color="mmc_category", color_discrete_map=CAT_COLORS)
fig.update_traces(
    marker=dict(cornerradius=5, line=dict(color=SURFACE, width=2)),
    tiling=dict(pad=3),
    texttemplate="%{label}<br>%{percentParent:.0%}", textfont=dict(size=13),
    hovertemplate="%{label}<br>%{value} messages<br>%{percentParent:.0%} of city<extra></extra>")

# px colours the CITY parent tiles by an aggregate, which borrows a category hue --
# override them to a neutral grey (category leaves keep their brand colour).
CITY_NEUTRAL = "#d9d7d2"
_cities = set(top5 + ["Other"])
fig.data[0].marker.colors = [CITY_NEUTRAL if lab in _cities else CAT_COLORS.get(lab, "#cccccc")
                             for lab in fig.data[0].labels]

fig.update_layout(
    title="Needs by category, city by city<br>"
          "<span style='font-size:13px;color:#8E8E78'>Grey block = city (Top 5 + Other), sized by volume · coloured tiles = category</span>",
    height=560, margin=dict(t=100, b=10, l=10, r=10))
fig.show()

### 5.2 Categories over time

In [24]:
# Weekly message volume, smoothed to a spline so the trend reads cleanly.
# NOTE on H3: whether demand peaks align with migration-policy announcements is
# NOT evaluated here -- we have no dataset of dated policy announcements to overlay.
# EVENTS is intentionally left empty; drop in real {"YYYY-MM-DD": "label"} milestones
# to test H3. Until then no policy-alignment claim is made from this chart.
EVENTS = {}   # intentionally empty: no verified policy-announcement dates available
weekly = msgs.dropna(subset=["ts"]).set_index("ts").resample("W").size()
yr = weekly.index.year.min()

fig = go.Figure()
fig.add_scatter(x=weekly.index, y=weekly.values, mode='lines+markers',
                line=dict(color=PRIMARY, width=2.5, shape='spline'),
                marker=dict(color=PRIMARY, size=6, line=dict(color=SURFACE, width=1)),
                fill='tozeroy', fillcolor='rgba(98,129,218,0.22)',
                hovertemplate="week of %{x|%B %d}<br>%{y} messages<extra></extra>")
for date, label in EVENTS.items():
    fig.add_vline(x=pd.Timestamp(date), line_dash='dash', line_color=MUTED,
                  annotation_text=label, annotation_position='top')
fig.update_layout(
    title=f"Weekly message volume <span style='color:#8E8E78'>({yr})</span>",
    xaxis=dict(title="", tickformat=DATE_FMT), yaxis=dict(title="Messages"),
    height=400, margin=dict(t=64))
fig.show()

In [25]:
# Top-4 categories over time (weekly), drawn as smoothed splines so the four
# trajectories separate cleanly.
cats4 = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
         .value_counts().head(4).index)
trend = (msgs.dropna(subset=["ts"])
         .assign(week=lambda d: d["ts"].dt.to_period("W").dt.start_time)
         [lambda d: d["mmc_category"].isin(cats4)]
         .groupby(["week", "mmc_category"]).size().unstack(fill_value=0)
         .reindex(columns=cats4, fill_value=0))
yr = trend.index.year.min()

fig = go.Figure()
for c in cats4:
    fig.add_scatter(x=trend.index, y=trend[c], mode='lines+markers', name=c,
                    line=dict(color=CAT_COLORS[c], width=2.5, shape='spline'),
                    marker=dict(color=CAT_COLORS[c], size=6,
                                line=dict(color=SURFACE, width=1)),
                    hovertemplate="%{x|%B %d}<br>" + c + ": %{y}<extra></extra>")
fig.update_layout(
    title=f"Top-4 categories over time (weekly) <span style='color:#8E8E78'>({yr})</span>",
    xaxis=dict(title="", tickformat=DATE_FMT), yaxis=dict(title="Messages"),
    legend=dict(orientation='h', y=-0.18, x=0), height=440, margin=dict(t=64))
fig.show()

### 5.3 Category mix by demographic (interactive)

One interactive chart instead of a wall of static treemaps. Use the **dropdown**
(top-right) to switch the demographic lens — **gender, city, nationality, age** —
and the 100%-stacked category mix repaints in place. Each bar is one group, split
into its category shares; colours are the shared brand category palette, so a
category keeps the same hue across every lens.

*(Built as an embedded Plotly control so it lives in the repo itself — no separate
Power BI file to keep in sync.)*

In [26]:
# §5.3 -- ONE interactive view instead of many static treemaps: the dropdown
# switches the demographic lens (gender / city / nationality / age) and the
# 100%-stacked category mix repaints in place. Category colours = shared CAT_COLORS,
# so each category keeps its hue across every lens.
top_cats = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]["mmc_category"]
            .value_counts().head(6).index.tolist())

DEMOS = [("Gender", "Gender", 4, GENDER_EN),
         ("City", "city_canon", 6, None),
         ("Nationality", "Nationality", 6, NATIONALITY_EN),
         ("Age range", "Age Ranges", 6, None)]

fig = go.Figure()
vis_blocks = []                                   # (start_index, n_traces) per lens
for col_lbl, col, topn, lab_map in DEMOS:
    base = msgs[msgs["mmc_category"].isin(top_cats) & msgs[col].notna()]
    groups = [g for g in base[col].value_counts().index if str(g) != "Otra"][:topn]
    ct = (pd.crosstab(base[col], base["mmc_category"], normalize="index") * 100)
    ct = ct.reindex(index=groups, columns=top_cats, fill_value=0).iloc[::-1]   # biggest on top
    ylabels = [en(lab_map, g) if lab_map else str(g) for g in ct.index]
    start = len(fig.data)
    for cat in top_cats:
        fig.add_bar(x=ct[cat].values, y=ylabels, orientation="h", name=cat,
                    marker=dict(color=CAT_COLORS.get(cat, NEGRO), line=dict(color=SURFACE, width=0.5)),
                    legendgroup=cat, visible=(col_lbl == "Gender"),
                    hovertemplate="%{y}<br>" + cat + ": %{x:.0f}%<extra></extra>")
    vis_blocks.append((start, len(top_cats)))

n_tr = len(fig.data)
def _title(lbl):
    return (f"Category mix by {lbl.lower()}<br><span style='font-size:13px;color:#8E8E78'>"
            "Share of each group's messages (100%) · switch lens with the dropdown</span>")
buttons = []
for i, (col_lbl, *_rest) in enumerate(DEMOS):
    mask = [False] * n_tr
    s, cnt = vis_blocks[i]
    for j in range(s, s + cnt):
        mask[j] = True
    buttons.append(dict(label=col_lbl, method="update",
                        args=[{"visible": mask}, {"title": _title(col_lbl)}]))

fig.update_layout(
    barmode="stack", title=_title("Gender"),
    xaxis=dict(title="Share of messages (%)", range=[0, 100]),
    yaxis=dict(title=""),
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right",
                      y=1.18, yanchor="top", showactive=True,
                      bgcolor="white", bordercolor="#8E8E78")],
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"),
    height=520, margin=dict(t=130, l=150))
fig.show()

### 5.4 Where users drop off, by category

In [27]:
# Where users stop, by category: one horizontal bar per category (the category of a
# user's last message), its length the number of users, split by gender. Sorted most
# -> fewest so the drop-off categories are easy to compare -- readable regardless of
# how lopsided the category sizes are.
pu = (msgs.groupby("phone")
      .agg(cat=("mmc_category", "first"), gender=("Gender", "first")).reset_index())
pu = pu[pu["cat"].isin(top_cats)]
counts = pu["cat"].value_counts()
cats = counts.index.tolist()[::-1]                 # smallest at top -> largest at bottom
genders = [g for g in pu["gender"].value_counts().index]
gcolor = {g: GENDER_COLOR.get(g, NEUTRAL) for g in genders}

fig = go.Figure()
for g in genders:
    xs = [int(((pu["cat"] == c) & (pu["gender"] == g)).sum()) for c in cats]
    fig.add_bar(y=cats, x=xs, orientation='h', name=en(GENDER_EN, g),
                marker=dict(color=gcolor[g], line=dict(color=SURFACE, width=1.2)),
                hovertemplate="%{y}<br>" + en(GENDER_EN, g) + ": %{x} users<extra></extra>")
# total-users label at the end of each bar
for c in cats:
    fig.add_annotation(x=int(counts[c]), y=c, text=f"  {int(counts[c])}", showarrow=False,
                       xanchor='left', font=dict(size=11, color=MUTED))
fig.update_layout(
    barmode='stack',
    title="Where users stop, by category<br>"
          "<span style='font-size:13px;color:#8E8E78'>Users by the category of their last message · split by gender</span>",
    xaxis=dict(title="Users", range=[0, counts.max() * 1.10]),
    yaxis=dict(title=""),
    legend=dict(title="Gender", orientation='h', y=-0.22, x=0, traceorder='normal'),
    height=470, margin=dict(l=180, t=90, b=90))
fig.show()
print(f"users ending on a courtesy message: "
      f"{(msgs[msgs['msg_idx']==msgs['n_msgs_user']-1]['mmc_category']==COURTESY).mean()*100:.1f}%")

users ending on a courtesy message: 0.0%


### 5.5 Satisfaction (MEAL) × category

> Only users who completed the MEAL survey appear (dozens), each assigned their
> dominant category and joined on phone. Read proportions as directional.

In [28]:
UTIL_ORDER = ["Nada útil", "Medianamente útil", "Útil", "Muy útil"]
dom_cat = (msgs[~msgs["mmc_category"].isin(NON_TOPIC_CATS)]
           .groupby("phone")["mmc_category"].agg(lambda s: s.value_counts().index[0])
           .rename("dom_cat").reset_index())
merged = dom_cat.merge(_meal_m[["phone", "utility"]], on="phone", how="inner")
merged = merged[merged["utility"].isin(UTIL_ORDER)]
print(f"users with both a category and a MEAL rating: {len(merged)}")

cats6 = merged["dom_cat"].value_counts().head(6).index.tolist()
sub = merged[merged["dom_cat"].isin(cats6)]
tabU = (pd.crosstab(sub["dom_cat"], sub["utility"], normalize="index")
        .reindex(index=cats6, columns=UTIL_ORDER, fill_value=0) * 100)
ramp = seq_hex(len(UTIL_ORDER))                    # light -> dark = least -> most useful

fig = go.Figure()
for i, u in enumerate(UTIL_ORDER):
    fig.add_bar(y=cats6, x=tabU[u].values, orientation='h', name=en(RATING_EN, u),
                marker=dict(color=ramp[i], line=dict(color=SURFACE, width=1.2)),
                hovertemplate="%{y}<br>" + en(RATING_EN, u) + ": %{x:.0f}%<extra></extra>")
fig.update_layout(
    barmode='stack',
    title="Utility rating by dominant category<br>"
          "<span style='font-size:13px;color:#8E8E78'>MEAL respondents · share of each category's users</span>",
    xaxis=dict(title="Share of respondents (%)", range=[0, 100], ticksuffix="%"),
    yaxis=dict(title="", autorange='reversed'),
    legend=dict(orientation='h', y=-0.30, x=0, traceorder='normal'),
    height=470, margin=dict(l=170, t=90, b=110))
fig.show()

users with both a category and a MEAL rating: 62


## 6. Engagement depth & abandonment

**What this shows.** How deeply users engage — what they ask about in general,
how many questions they send, how many were surveyed, and how far the typical
user goes before stopping (**H5**). The bridge to Notebook 3.

### 6.1 Topics discussed — simple frequency

In [29]:
# What topics are users asking about? Chat_summary is free-text/LLM-generated with
# several inconsistencies (leftover prompt text, "''" separators, hashtags, mixed
# case); TOPIC_MAP normalizes down to 7 canonical categories.
TOPIC_MAP = {
    'legal documentation': 'Legal Documentation',
    'humanitarian assistance': 'Humanitarian Assistance',
    'employment': 'Employment', 'services': 'Services', 'protection': 'Protection',
    'organization search': 'Organization Search',
    'journey information': 'Journey Information',
}
raw = df['Chat_summary'].dropna()
raw = raw[~raw.str.contains('Use exactly one of these hashtags', na=False)]
topics = (
    raw.str.replace("''", ',', regex=False).str.split(',').explode()
    .str.strip().str.lstrip('#').str.replace('_', ' ', regex=False).str.lower()
    .map(TOPIC_MAP).dropna().value_counts()
)
order = topics.iloc[::-1]

fig = go.Figure(go.Bar(
    x=order.values, y=order.index, orientation='h',
    marker=dict(color=seq_by_value_hex(order.values), line=dict(color=SURFACE, width=0.5)),
    text=order.values, textposition='outside', cliponaxis=False,
    hovertemplate="%{y}<br>%{x} mentions<extra></extra>"))
fig.update_layout(
    title="What do users ask Sami about? Topics discussed",
    xaxis=dict(title="Mentions", range=[0, topics.max() * 1.12]),
    yaxis=dict(title=""), height=420, margin=dict(l=190, t=64))
fig.show()

### 6.2 Questions per user

In [30]:
# How many questions does a typical user ask? A histogram isn't useful (almost
# everyone asks 0-4), so a summary + distribution table communicates it better.
qs = df['Questions per user'].dropna()
print("Questions per user — summary statistics:")
print(qs.describe().round(2).to_string())
print()
counts = qs.value_counts().sort_index()
summary = pd.DataFrame({
    'Questions asked': counts.index.astype(int),
    'Users': counts.values,
    'Share of users (%)': (counts.values / counts.sum() * 100).round(1),
})
print("Distribution by number of questions asked:")
print(summary.to_string(index=False))

Questions per user — summary statistics:
count    942.00
mean       1.59
std        2.18
min        0.00
25%        0.00
50%        1.00
75%        2.00
max       29.00

Distribution by number of questions asked:
 Questions asked  Users  Share of users (%)
               0    325                34.5
               1    265                28.1
               2    160                17.0
               3     82                 8.7
               4     45                 4.8
               5     17                 1.8
               6     17                 1.8
               7     10                 1.1
               8      8                 0.8
               9      5                 0.5
              10      1                 0.1
              11      2                 0.2
              14      1                 0.1
              15      3                 0.3
              29      1                 0.1


### 6.3 Survey reach

In [31]:
# What fraction of users were sent the post-conversation survey?
survey = df['Survey sent'].fillna('Not sent').value_counts()
labels = [en(SURVEY_EN, str(k)).capitalize() for k in survey.index]
colors = [MUTED if 'not' in l.lower() else PRIMARY for l in labels]  # Sent = blue
total = int(survey.sum())

fig = go.Figure(go.Pie(
    labels=labels, values=survey.values, hole=0.55, sort=False,
    direction='clockwise',
    marker=dict(colors=colors, line=dict(color=SURFACE, width=2)),
    textinfo='label+percent', textposition='outside',
    hovertemplate="%{label}<br>%{value} users (%{percent})<extra></extra>"))
fig.update_layout(
    title="Was the follow-up survey sent?",
    annotations=[dict(text=f"{total}<br>users", x=0.5, y=0.5, showarrow=False,
                      font=dict(size=18, color=INK))],
    showlegend=False, height=440, margin=dict(t=64))
fig.show()

### 6.4 Messages per user (depth)

In [32]:
# How deep does engagement go? Messages per user, each bar shaded light->dark by
# depth (the deeper the conversation, the darker the bar).
per_user = msgs.groupby("phone")["n_msgs_user"].first()
XMAX = 20
vc = per_user.clip(upper=XMAX).value_counts().sort_index()
depths = vc.index.to_numpy()
counts = vc.values

fig = go.Figure(go.Bar(
    x=depths, y=counts,
    marker=dict(color=seq_by_value_hex(depths), line=dict(color=SURFACE, width=0.6)),
    hovertemplate="%{x} messages<br>%{y} users<extra></extra>"))
med = per_user.median()
fig.add_vline(x=med, line_dash='dash', line_color=MUTED,
              annotation_text=f"median {med:.0f}", annotation_position='top right',
              annotation_font=dict(color=MUTED, size=11))
fig.update_layout(
    title="How deep does engagement go? Messages per user<br>"
          "<span style='font-size:13px;color:#8E8E78'>Bar shade = conversation depth</span>",
    xaxis=dict(title="Messages per user", dtick=1, range=[0.5, XMAX + 0.5]),
    yaxis=dict(title="Users"), height=430, margin=dict(t=90))
fig.show()
print(f"median {per_user.median():.0f}  |  single-message share "
      f"{(per_user<=1).mean()*100:.1f}%  |  90th pct {per_user.quantile(.9):.0f}")

median 3  |  single-message share 24.8%  |  90th pct 7
